In [68]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, f1_score
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [71]:
data = pd.read_csv('Ora_data.csv')

In [73]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

def evaluate_model(model, X, y, dataset_name):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)
    
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred, zero_division=0)
    
    print(f"\n--- {dataset_name} Set Results ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y, y_pred))
    
    # Calculate metrics at lower thresholds for comparison
    print("\nMetrics at different probability thresholds:")
    for threshold in [0.1, 0.2, 0.3, 0.4]:
        y_pred_at_threshold = (y_pred_proba >= threshold).astype(int)
        f1_at_threshold = f1_score(y, y_pred_at_threshold, zero_division=0)
        precision_at_threshold = precision_score(y, y_pred_at_threshold, zero_division=0)
        recall_at_threshold = recall_score(y, y_pred_at_threshold)
        print(f"  Threshold {threshold:.1f} - F1: {f1_at_threshold:.4f}, "
              f"Precision: {precision_at_threshold:.4f}, Recall: {recall_at_threshold:.4f}")

# Load the engineered and encoded dataset
# This assumes data is your prepared dataset
y = data['churn_target']
X = data.drop(columns=['churn_target'])

# Create train, validation, and test sets (60%, 20%, 20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train set: {X_train.shape[0]} samples, Churn rate: {y_train.mean():.4f}")
print(f"Validation set: {X_val.shape[0]} samples, Churn rate: {y_val.mean():.4f}")
print(f"Test set: {X_test.shape[0]} samples, Churn rate: {y_test.mean():.4f}")

# Based on your results, this is the model we used
model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=8,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,  
    reg_lambda=0.1,  
    random_state=42,
    class_weight='balanced',  # or is_unbalance=True
    objective='binary',
    metric='binary_logloss'
)

# Fit the model
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc'
    )

# Evaluate on all three sets
print("\nEvaluating model...")
evaluate_model(model, X_train, y_train, "Training")
evaluate_model(model, X_val, y_val, "Validation")
evaluate_model(model, X_test, y_test, "Test")

# Feature importance analysis
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
})
print("\nTop 15 Feature Importances:")
print(feature_importance.sort_values('Importance', ascending=False).head(15).to_string(index=False))

Train set: 7921 samples, Churn rate: 0.0646
Validation set: 2641 samples, Churn rate: 0.0647
Test set: 2641 samples, Churn rate: 0.0647

Evaluating model...

--- Training Set Results ---
Precision: 0.2259
Recall: 0.9824
F1 Score: 0.3673

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.77      0.87      7409
           1       0.23      0.98      0.37       512

    accuracy                           0.78      7921
   macro avg       0.61      0.87      0.62      7921
weighted avg       0.95      0.78      0.84      7921

Confusion Matrix:
[[5685 1724]
 [   9  503]]

Metrics at different probability thresholds:
  Threshold 0.1 - F1: 0.1380, Precision: 0.0741, Recall: 1.0000
  Threshold 0.2 - F1: 0.1667, Precision: 0.0910, Recall: 1.0000
  Threshold 0.3 - F1: 0.2060, Precision: 0.1148, Recall: 1.0000
  Threshold 0.4 - F1: 0.2664, Precision: 0.1537, Recall: 0.9980

--- Validation Set Results ---
Precision: 0.0871
Recall: 0.3450
